# 3.3 Human-in-the-Loop (adaptado a lab local)

**Qué se ve aquí:** `HumanInTheLoopMiddleware` pausa el agente **antes** de ejecutar ciertas tools (`interrupt_on`). La respuesta trae `__interrupt__` y se reanuda con `Command(resume={"decisions": [...]})` usando **el mismo `thread_id`** (por eso hace falta un checkpointer). Decisiones: `approve`, `reject` (con mensaje) y `edit` (cambiando los argumentos).

> **Nota local:** cada decisión consume el interrupt pendiente. Para probar *reject* y *edit*, vuelve a correr la celda de `agent.invoke(...)` inicial antes de cada una (si no, no hay nada que reanudar).

**Convención de este fork:** 🔸 ORIGINAL DEL CURSO (comentada, con el motivo) → 🟢 ADAPTADO LOCAL (la que corre con Ollama). Celdas sin marca = iguales al curso.

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: falta cargar local_model.py (raíz del repo) y definir el modelo local.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from dotenv import load_dotenv
#
# load_dotenv()

In [ ]:
# 🟢 ADAPTADO LOCAL
# Motivo: el notebook está en notebooks/module-3/ y local_model.py en la raíz del repo.
#         MODEL reemplaza a "gpt-5-nano" en todas las celdas (gemma4: soporta tools, el más rápido del lab).
import sys, pathlib
sys.path.insert(0, str(next(p for p in [pathlib.Path.cwd(), *pathlib.Path.cwd().parents] if (p / "local_model.py").exists())))

from dotenv import load_dotenv
load_dotenv()
from local_model import get_model, OLLAMA_URL

MODEL = get_model("gemma4:latest")

In [ ]:
# Sin cambios
from langchain.tools import tool, ToolRuntime

@tool
def read_email(runtime: ToolRuntime) -> str:
    """Read an email from the given address."""
    # take email from state
    return runtime.state["email"]

@tool
def send_email(body: str) -> str:
    """Send an email to the given address with the given subject and body."""
    # fake email sending
    return f"Email sent"

In [ ]:
# 🔸 ORIGINAL DEL CURSO — no se corre en el lab local
# Motivo: "gpt-5-nano" es API de pago.
# Ver la celda siguiente (🟢 ADAPTADO LOCAL).
#
# from langchain.agents import create_agent, AgentState
# from langgraph.checkpoint.memory import InMemorySaver
# from langchain.agents.middleware import HumanInTheLoopMiddleware
#
# class EmailState(AgentState):
#     email: str
#
# agent = create_agent(
#     model="gpt-5-nano",
#     tools=[read_email, send_email],
#     state_schema=EmailState,
#     checkpointer=InMemorySaver(),
#     middleware=[
#         HumanInTheLoopMiddleware(
#             interrupt_on={
#                 "read_email": False,
#                 "send_email": True,
#             },
#             description_prefix="Tool execution requires approval",
#         ),
#     ],
# )

In [ ]:
# 🟢 ADAPTADO LOCAL
# Motivo: "gpt-5-nano" es API de pago.
from langchain.agents import create_agent, AgentState
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents.middleware import HumanInTheLoopMiddleware

class EmailState(AgentState):
    email: str

agent = create_agent(
    model=MODEL,
    tools=[read_email, send_email],
    state_schema=EmailState,
    checkpointer=InMemorySaver(),
    middleware=[
        HumanInTheLoopMiddleware(
            interrupt_on={
                "read_email": False,
                "send_email": True,
            },
            description_prefix="Tool execution requires approval",
        ),
    ],
)

In [ ]:
# Sin cambios
from langchain.messages import HumanMessage

config = {"configurable": {"thread_id": "1"}}

response = agent.invoke(
    {
        "messages": [HumanMessage(content="Please read my email and send a response immediately. Send the reply now in the same thread.")],
        "email": "Hi Seán, I'm going to be late for our meeting tomorrow. Can we reschedule? Best, John."
    },
    config=config
)

In [ ]:
# Sin cambios
from pprint import pprint

pprint(response)

In [ ]:
# Sin cambios
print(response['__interrupt__'])

In [ ]:
# Sin cambios
# Access just the 'body' argument from the tool call
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

## Approve

In [ ]:
# Sin cambios
from langgraph.types import Command

response = agent.invoke(
    Command( 
        resume={"decisions": [{"type": "approve"}]}
    ), 
    config=config # Same thread ID to resume the paused conversation
)

pprint(response)

## Reject

In [ ]:
# Sin cambios
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "reject",
                    # An explanation of why the request was rejected
                    "message": "No please sign off - Your merciful leader, Seán."
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)

In [ ]:
# Sin cambios
print(response['__interrupt__'][0].value['action_requests'][0]['args']['body'])

## Edit

In [ ]:
# Sin cambios
response = agent.invoke(
    Command(        
        resume={
            "decisions": [
                {
                    "type": "edit",
                    # Edited action with tool name and args
                    "edited_action": {
                        # Tool name to call.
                        # Will usually be the same as the original action.
                        "name": "send_email",
                        # Arguments to pass to the tool.
                        "args": {"body": "This is the last straw, you're fired!"},
                    }
                }
            ]
        }
    ), 
    config=config # Same thread ID to resume the paused conversation
    )   

pprint(response)